In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
solar_api = os.getenv('SOLAR_API')



In [2]:
prompt = """
너는 기업 사업보고서의 "주요 제품", "주요 제품 및 서비스", "주요제품등의 현황" 표에서
매출 구성(금액/비중)을 정확히 추출한다.

절대 원칙
- 표에 없는 숫자 생성 금지
- 다른 행 숫자 복사 금지
- 숫자는 해당 행 셀에서만 읽기

────────────────
1. 컬럼 식별
────────────────
표에서 다음 역할을 찾는다 (순서 무관)

사업부문 (있으면 사용)
매출유형 (제품 / 상품 / 기타 등)
품목 (제품명 / 서비스명)
구체적용도 (품목 보조)
주요상표 (품목 보조)
매출액
비율(%)

────────────────
2. 품목 결정
────────────────
항목 키는 반드시 품목.

품목이 비어있으면 다음 순서로 복원:
1 구체적용도
2 주요상표
3 병합된 직전 품목
직전 품목 계승은
셀 병합이 명확히 확인될 때만 허용한다.

행이 바뀌었거나
매출유형이 바뀌었으면
품목 계승 금지.

그래도 없으면 해당 행 제외.

"기타"는 매출유형이 아니라 품목일 수 있음 → 품목으로 유지.

────────────────
3. 병합셀 처리
────────────────
텍스트만 계승 가능.

사업부문 비어있으면 직전 사업부문 계승
매출유형 비어있으면 직전 매출유형 계승

단, 사업부문 변경 시 초기화.

숫자(금액/비율)는 절대 계승 금지.

────────────────
4. 값 추출
────────────────
금액 공란 → 0
비율 공란 → "0.00%"

금액은 정수
비율은 소수 둘째자리

────────────────
5. 합계
────────────────
합계 행은 별도 저장.
여러 소계 존재 시 최종 합계만.

합계 없으면 금액 합으로 계산.

────────────────
6. 기간 식별
────────────────

표에서 기간 정보를 우선적으로 식별한다.

기간 식별 우선순위:

1순위: 기수 정보 (예: 제23기, 제25기 등)
→ 여러 기수 존재하면 가장 큰 기수 사용

2순위: 연도 정보 (예: 2024년도, 2023년 등)
→ 여러 연도 존재하면 가장 최신 연도 사용

3순위: 기간 정보 없음
→ "기간미상" 사용

주의:
"연결재무제표 기준", "별도기준" 등은
회계 기준일 뿐 기간이 아니다.
기간으로 사용하지 않는다.

연도 범위(YYYY.MM.DD~YYYY.MM.DD)는 종료연도 사용

────────────────
7. 중요 구조 규칙
────────────────
매출유형은 합계 계산 단위가 아니다.
모든 품목은 하나의 전체 매출 집합에 속한다.

즉:
매출유형별 부분 합계는 계산하지 않는다.
전체 품목을 합친 하나의 전체 합계만 존재한다.

매출유형은 단순 분류 계층이다.
────────────────
8. 출력 (JSON만)
────────────────
{
 "<기간>": {
   "<매출유형>": {
     "<품목>": {"금액": 정수, "비중": "0.00%"}
   },
   "합계": {"금액": 정수, "비중": "100.00%"}
 }
}

매출유형은 하위 분류이며
모든 매출유형은 동일한 전체 합계를 공유한다.
매출유형별 개별 합계를 만들지 않는다.

"""

In [3]:
#prompt = ""


In [4]:
content ="""

2. 주요 제품 및 서비스


가. 주요 제품 등의 현황

(단위 : 천원,%)
사업부문	매출유형	품   목	구체적용도	주요상표등	매출액	비율
피엔풍년	제 품	압력솥외	주방용품	풍년	 19,936,001	36.3%
상 품	압력솥외	주방용품	풍년	 32,326,182	58.8%
부 품	부 품	A/S 용도	-	  1,638,816	3.0%
임대료	-	-	-	    123,469	0.2%
기타	-	-	-	    130,253	0.3%
피엔랩	상품	전기가전	주방용품	모노	    781,205	1.4%
기타	-	-	-	      12,084	0.0%
합 계	-	-	54,948,010	100.0%
※ 상기 실적은 연결재무제표 기준입니다.


나. 주요 제품 등의 가격변동추이

(단위 : 원)
품 목	제51기	제50기	제49기
압력솥류	내수	            78,387	78,157	61,106
수출	            111,673	120,288	86,127
전기밥솥/전기류	내수	             45,769	37,124	25,161
수출	            114,643	82,688	45,475
기물류	내수	              18,626	17,595	20,454
수출	              27,820	26,792	27,688
(1)산출기준
- 매출액(세금계산서상) / 판매수량


(2) 주요 가격변동원인

- 당기에는 전기류(수출)실적이 미비하여 품목별 단가 차이가 큼.




"""

In [5]:

client = OpenAI(
    api_key=solar_api,
    base_url="https://api.upstage.ai/v1"
)

response = client.chat.completions.create(
    model="solar-pro3",
    messages=[
        {'role': 'system', 'content': prompt},
        {"role": "user", "content": content}
    ],
    temperature=0,
    max_tokens=65536,
    reasoning_effort="high",
)
print(response.choices[0].message.content)

{
  "제51기": {
    "제 품": {
      "압력솥외 (제 품)": {"금액": 19936001, "비중": "36.30%"},
      "압력솥외 (상 품)": {"금액": 32326182, "비중": "58.80%"}
    },
    "상 품": {
      "압력솥외 (상 품)": {"금액": 32326182, "비중": "58.80%"}
    },
    "부 품": {
      "A/S 용도 (부 품)": {"금액": 1638816, "비중": "3.00%"}
    },
    "임대료": {
      "A/S 용도 (임대료)": {"금액": 123469, "비중": "0.20%"}
    },
    "기타": {
      "A/S 용도 (기타)": {"금액": 130253, "비중": "0.30%"}
    },
    "상품": {
      "전기가전 (상품)": {"금액": 781205, "비중": "1.40%"}
    },
    "기타": {
      "전기가전 (기타)": {"금액": 12084, "비중": "0.00%"}
    },
    "합계": {"금액": 54948010, "비중": "100.00%"}
  }
}
